In [87]:
import duckdb
import pandas as pd

con = duckdb.connect()

# Turnover Study
**Offensive Rebounds are currently worth 1 in System Score, investigation into alternative methods of counting and thus incentivizing "More" is done through looking at turnover added-value.**

The file below contains additional data tracked after the season ended with a 'Type' column for both Home and Away collected by rewatching all 141 games from the Mountain East Conference season.  
In the 'Type' column(s):
- Turnovers are tracked via:
    - LF: Full-Court Live-Ball Turnovers Committed
    - LH: Half-Court Live-Ball Turnovers Committed
    - DF: Full-Court Dead-Ball Turnovers Committed
    - DH: Half-Court Dead-Ball Turnovers Committed

In [88]:
file = "Fouls_and_TOs.csv"

In [89]:
data_q = f"""
    SELECT Row_ID, Date, Home_Team, Home_Type, Home_Points, Away_Team, Away_Type, Away_Points, Period FROM {file}
    WHERE Date IS NOT NULL
"""
data = con.query(data_q)

In [90]:
data

┌────────┬────────────┬────────────┬───────────┬─────────────┬───────────┬───────────┬─────────────┬────────┐
│ Row_ID │    Date    │ Home_Team  │ Home_Type │ Home_Points │ Away_Team │ Away_Type │ Away_Points │ Period │
│ int64  │    date    │  varchar   │  varchar  │    int64    │  varchar  │  varchar  │    int64    │ int64  │
├────────┼────────────┼────────────┼───────────┼─────────────┼───────────┼───────────┼─────────────┼────────┤
│      1 │ 2025-11-25 │ Fairmont   │ LF        │           0 │ WVSU      │ F3        │           3 │      1 │
│      2 │ 2025-11-25 │ Fairmont   │ NULL      │           0 │ WVSU      │ NULL      │           0 │      1 │
│      3 │ 2025-11-25 │ Fairmont   │ NULL      │           0 │ WVSU      │ NULL      │           0 │      1 │
│      4 │ 2025-11-25 │ Fairmont   │ NULL      │           0 │ WVSU      │ NULL      │           2 │      1 │
│      5 │ 2025-11-25 │ Fairmont   │ NULL      │           2 │ WVSU      │ LH        │           0 │      1 │
│      6 │

## This code gives individual Team, Type of Turnover, and Points off Turnover ##
- Data is organized Row # | Date | Home Team | Home Type | Home Points | Away Team | Away Type | Away Points with numerous intermediary stats in between each Type and Points columns
- Therefore, to calculate the points *allowed* following turnover:
   1. Home Team: Simply use Away Points in same row
   2. Away Team: Use Home Points as long as Date, Home Team, and Period from current row and next row match

This query gives the row by row values only when a turnover occurs.

In [157]:
to_points_q = f"""
    WITH ordered AS (
        SELECT *,
            LEAD(Home_Points) OVER (ORDER BY Row_ID) AS Next_Home_Points,
            LEAD(Date)        OVER (ORDER BY Row_ID) AS Next_Date,
            LEAD(Home_Team)   OVER (ORDER BY Row_ID) AS Next_Home_Team,
            LEAD(Period)      OVER (ORDER BY Row_ID) AS Next_Period
        FROM data
    ),
    sorted AS (SELECT
        Row_ID, Date, Home_Team, Home_Type, Home_Points,
        Away_Team, Away_Type, Away_Points, Period,
        CASE WHEN Home_Type IN ('LF','LH','DF','DH')
             THEN Away_Points
        END AS Home_Points_Off_TO,
        CASE WHEN Away_Type IN ('LF','LH','DF','DH')
             AND Date = Next_Date
             AND Home_Team = Next_Home_Team
             AND Period = Next_Period
        THEN Next_Home_Points
        END AS Away_Points_Off_TO
    FROM ordered)
    SELECT Row_ID, Date, Home_Team as Team, Home_Type as Type, Home_Points_Off_TO as Points_Off_TO
    FROM sorted
    WHERE Home_Type IS NOT NULL AND Home_Type NOT IN ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    UNION ALL
    SELECT Row_ID, Date, Away_Team as Team, Away_Type as Type, Away_Points_Off_TO as Points_Off_TO
    FROM SORTED
    Where Away_Type IS NOT NULL AND Away_Type NOT IN ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    ORDER BY Row_ID
"""
con.sql(to_points_q)

┌────────┬────────────┬──────────────┬─────────┬───────────────┐
│ Row_ID │    Date    │     Team     │  Type   │ Points_Off_TO │
│ int64  │    date    │   varchar    │ varchar │     int64     │
├────────┼────────────┼──────────────┼─────────┼───────────────┤
│      1 │ 2025-11-25 │ Fairmont     │ LF      │             3 │
│      5 │ 2025-11-25 │ WVSU         │ LH      │             3 │
│      7 │ 2025-11-25 │ WVSU         │ LF      │             2 │
│     10 │ 2025-11-25 │ WVSU         │ LH      │             2 │
│     21 │ 2025-11-25 │ Fairmont     │ LH      │             0 │
│     23 │ 2025-11-25 │ Fairmont     │ LH      │             0 │
│     25 │ 2025-11-25 │ WVSU         │ DH      │             0 │
│     32 │ 2025-11-25 │ WVSU         │ DH      │             3 │
│     37 │ 2025-11-25 │ WVSU         │ DF      │             2 │
│     39 │ 2025-11-25 │ WVSU         │ DF      │             2 │
│      · │     ·      │  ·           │ ·       │             · │
│      · │     ·      │  

In [123]:
totals = f"""
    SELECT Type, CAST(AVG(Points_Off_TO) AS DECIMAL(10, 2)) as PPP, SUM(Points_Off_TO) as Points_Off_TO, COUNT(Points_Off_TO) as Scoring_Poss
    FROM ({to_points_q})
    WHERE Points_Off_TO IS NOT NULL
    GROUP BY Type
    ORDER BY PPP DESC
"""
print(con.sql(totals))

live_v_dead = f"""
    WITH l_v_d AS (
        SELECT * FROM
        VALUES ('LH', 'Live'), ('LF', 'Live'), ('DH', 'Dead'), ('DF', 'Dead')
        AS t('Type', 'Live_or_Dead')
    ),
    comb AS (
        SELECT t.*, ld.Live_or_Dead
        FROM ({totals}) as t
        FULL JOIN l_v_d as ld
        ON t.Type = ld.Type
    )
    SELECT Live_or_Dead, SUM(Points_Off_TO) as Points, SUM(Scoring_Poss) as Scoring_Poss, ROUND(SUM(Points_Off_TO)/SUM(Scoring_Poss), 2) as PPP
    FROM comb
    GROUP BY Live_or_Dead
"""

con.sql(live_v_dead)

┌─────────┬───────────────┬───────────────┬──────────────┐
│  Type   │      PPP      │ Points_Off_TO │ Scoring_Poss │
│ varchar │ decimal(10,2) │    int128     │    int64     │
├─────────┼───────────────┼───────────────┼──────────────┤
│ LF      │          1.38 │           425 │          308 │
│ LH      │          1.30 │          2243 │         1720 │
│ DF      │          1.09 │           201 │          184 │
│ DH      │          1.04 │          1365 │         1313 │
└─────────┴───────────────┴───────────────┴──────────────┘



┌──────────────┬────────┬──────────────┬────────┐
│ Live_or_Dead │ Points │ Scoring_Poss │  PPP   │
│   varchar    │ int128 │    int128    │ double │
├──────────────┼────────┼──────────────┼────────┤
│ Live         │   2668 │         2028 │   1.32 │
│ Dead         │   1566 │         1497 │   1.05 │
└──────────────┴────────┴──────────────┴────────┘

Looking at the two tables of gives some indication of the power of turnovers.
1. The gap between live-ball and dead-ball is more than noticeable. It has an approximate gap of 0.25 (which is a 1 point equivalent in the Scoring System!) This is a early (yet way-too-early) sign that specifically live balls can be used to add to the system scoring.
2. What's interesting is that dead-ball turnovers in the half-court are not too far different from dead-balls in the full court (and similarly for live-ball). Perhaps this is due to the shot clock reset only being 20 seconds when turnovers occur in the full court.
3. The sample sizes for full-court turnovers whether looking at dead-ball or live-ball are not too big, but not necessarily small either. Something to be wary of.

## Breaking Down Turnover Types by Team ##

In [137]:
turnovers = f"""
    SELECT Team, Type, ROUND(AVG(Points_Off_TO), 2) as PPP, COUNT(Points_Off_TO) AS Scoring_Poss
    FROM ({to_points_q})
    WHERE Points_Off_TO NOT NULL
    GROUP BY Team, Type
    ORDER BY Team, Type
"""

#con.sql(turnovers).df()

In [172]:
to_points_q_2 = f"""
    WITH ordered AS (
        SELECT *,
            LAG(Date)        OVER (ORDER BY Row_ID) AS Prev_Date,
            LAG(Home_Team)   OVER (ORDER BY Row_ID) AS Prev_Home_Team,
            LAG(Period)      OVER (ORDER BY Row_ID) AS Prev_Period,
            LAG(Away_Type) OVER (ORDER BY Row_ID) AS Prev_Away_Type
        FROM data
    ),
    sorted AS (SELECT
        Row_ID, Date, Home_Team, Home_Type, Home_Points,
        Away_Team, Prev_Away_Type, Away_Type, Away_Points, Period,
        CASE WHEN Home_Type IN ('LF','LH','DF','DH')
             THEN Away_Points
        END AS Away_Points_Off_TO,
        CASE WHEN Prev_Away_Type IN ('LF','LH','DF','DH')
             AND Date = Prev_Date
             AND Home_Team = Prev_Home_Team
             AND Period = Prev_Period
        THEN Home_Points
        END AS Home_Points_Off_TO
    FROM ordered)
    SELECT Row_ID, Away_Team as Team, Home_Type as Type, Away_Points_Off_TO as Points_Off_TO
    FROM sorted
    WHERE Home_Type IS NOT NULL AND Home_Type NOT IN ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    UNION ALL
    SELECT Row_ID, Home_Team as Team, Prev_Away_Type as Type, Home_Points_Off_TO as Points_Off_TO
    FROM sorted
    WHERE Prev_Away_Type IS NOT NULL AND Prev_Away_Type NOT IN ('T1', 'T2', 'F2', 'F2A', 'F3', 'F3A', 'OAO', 'DB')
    ORDER BY Row_ID
"""
turnovers = f"""
    SELECT Team, Type, SUM(Points_Off_TO) as Points_Off_TO, COUNT(Points_Off_TO) AS Scoring_Poss, ROUND(AVG(Points_Off_TO), 2) as PPP
    FROM ({to_points_q_2})
    WHERE Points_Off_TO IS NOT NULL
    GROUP BY Team, Type
    ORDER BY Scoring_Poss DESC
"""

live_v_dead = f"""
    WITH l_v_d AS (
        SELECT * FROM
        VALUES ('LH', 'Live'), ('LF', 'Live'), ('DH', 'Dead'), ('DF', 'Dead')
        AS t('Type', 'Live_or_Dead')
    ),
    games AS (
        SELECT Team, COUNT(DISTINCT Date) AS Games
        FROM ({to_points_q})
        GROUP BY Team
    ),
    comb AS (
        SELECT t.*, ld.Live_or_Dead
        FROM ({turnovers}) as t
        FULL JOIN l_v_d as ld
        ON t.Type = ld.Type
    )
    SELECT c.Team, c.Live_or_Dead, SUM(c.Points_Off_TO) as Points, SUM(c.Scoring_Poss) as Scoring_Poss, ROUND(SUM(c.Points_Off_TO)/SUM(c.Scoring_Poss), 2) as PPP, MAX(g.Games) as Games, ROUND(SUM(c.Scoring_Poss)/MAX(g.Games), 2) as Scoring_Poss_PG
    FROM comb c
    JOIN Games g ON c.Team = g.Team
    GROUP BY c.Team, c.Live_or_Dead
    ORDER BY Scoring_Poss_PG DESC
    
"""


print(con.sql(turnovers))
con.sql(live_v_dead)

┌──────────────┬─────────┬───────────────┬──────────────┬────────┐
│     Team     │  Type   │ Points_Off_TO │ Scoring_Poss │  PPP   │
│   varchar    │ varchar │    int128     │    int64     │ double │
├──────────────┼─────────┼───────────────┼──────────────┼────────┤
│ Charleston   │ LH      │           230 │          165 │   1.39 │
│ West Liberty │ LH      │           217 │          164 │   1.32 │
│ Glenville    │ LH      │           239 │          161 │   1.48 │
│ Wheeling     │ LH      │           204 │          150 │   1.36 │
│ Frostburg    │ LH      │           215 │          149 │   1.44 │
│ Fairmont     │ LH      │           208 │          148 │   1.41 │
│ Wesleyan     │ LH      │           160 │          142 │   1.13 │
│ West Liberty │ LF      │           200 │          141 │   1.42 │
│ Glenville    │ DH      │           152 │          136 │   1.12 │
│ D&E          │ LH      │           167 │          132 │   1.27 │
│  ·           │ ·       │             · │            · │     

┌──────────────┬──────────────┬────────┬──────────────┬────────┬───────┬─────────────────┐
│     Team     │ Live_or_Dead │ Points │ Scoring_Poss │  PPP   │ Games │ Scoring_Poss_PG │
│   varchar    │   varchar    │ int128 │    int128    │ double │ int64 │     double      │
├──────────────┼──────────────┼────────┼──────────────┼────────┼───────┼─────────────────┤
│ West Liberty │ Live         │    417 │          305 │   1.37 │    25 │            12.2 │
│ Charleston   │ Live         │    245 │          180 │   1.36 │    23 │            7.83 │
│ West Liberty │ Dead         │    208 │          189 │    1.1 │    25 │            7.56 │
│ Wheeling     │ Live         │    233 │          170 │   1.37 │    23 │            7.39 │
│ Frostburg    │ Live         │    255 │          174 │   1.47 │    24 │            7.25 │
│ Fairmont     │ Live         │    249 │          179 │   1.39 │    25 │            7.16 │
│ Wesleyan     │ Live         │    183 │          157 │   1.17 │    22 │            7.14 │

Looking at the turnovers earlier, creating live-ball appeared to be superior to creating dead-ball. The breakdown by team helps us see that even further...
1. Charleston created the most turnovers of any of the four possible breakdowns with 165 live-ball turnovers forced in the half-court. This was done in playing in 2 less games than West Liberty and Fairmont, 1 less than Game than Glenville and Concord.
2. When it comes to creating live-ball turnovers, West Liberty dwarfs everyone with 12.2 TO/G (resulting in Scoring Possessions) and comes in 3rd in creating dead-ball turnovers with 7.56 TO/G
3. Charleston's creation of live-ball turnovers came in second with 7.83 TO/G
4. It is no wonder that West Liberty has success. It is well known that their press has turnover effects, but it just looms so much larger over everyone else's...